<a href="https://colab.research.google.com/github/johnmichaels4867-create/ACCTG5150/blob/main/Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
f = '/content/RideSafeCycleworks_GL_FY2025.xlsx'
gl = pd.read_excel(f, "GL_Detail", dtype={"GL_Account_Number": str})
tb = pd.read_excel(f, "Trial_Balance", dtype={"GL_Account_Number": str})

In [19]:
gl["AmountSigned"] = gl["Amount"] * gl["Amount_Credit_Debit_Indicator"].map({"D": 1, "C": -1})

# 4.1 Aggregate GL to trial balance grain
keys = ["GL_Account_Number", "Business_Unit_Code", "Period"]

gl_activity = (
    gl
    .groupby(keys, as_index=False)
    .agg(Activity=("AmountSigned", "sum"), Line_Count=("AmountSigned", "size"))
)

# Merge TB with GL activity
rf = tb.merge(
    gl_activity,
    on=keys,
    how="left",
    indicator=True,          # adds a _merge column: "both" or "left_only"
    validate="one_to_one",   # errors out if either side has duplicate keys
)
print(rf["_merge"].value_counts())   # right_only is always 0 in a left merge

# Fill unmatched rows with 0 (no GL activity that period)
rf[["Activity", "Line_Count"]] = rf[["Activity", "Line_Count"]].fillna(0)

# Calculate Diff
rf["Diff"] = (rf["Amount_Ending"] - rf["Amount_Beginning"] - rf["Activity"]).round(2)
print("# of rows that don't tie:", (rf["Diff"] != 0).sum())

_merge
both          1995
left_only      761
right_only       0
Name: count, dtype: int64
# of rows that don't tie: 0


In [31]:
rf[["GL_Account_Number", "Business_Unit_Code", "Period", "Diff"]].head()

,GL_Account_Number,Business_Unit_Code,Period,Diff
0,10100,RSC-CORP,M01,0.0
1,10100,RSC-CORP,M02,-0.0
2,10100,RSC-CORP,M03,0.0
3,10100,RSC-CORP,M04,-0.0
4,10100,RSC-CORP,M05,-0.0


In [21]:
pivot = rf.pivot_table(index="GL_Account_Number", columns="Period", values="Diff", aggfunc="sum")

In [35]:
pivot.head()

Period,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11,M12,M13
GL_Account_Number,,,,,,,,,,,,,
10100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10200,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
# Balances with no activity (TB rows with no GL match)
# already in rf, where Line_Count is 0
tb_no_activity = rf[rf["Line_Count"] == 0]

# Activity with no balance (GL rows with no TB match)
gl_check = gl_activity.merge(tb, on=keys, how="left", indicator=True)
gl_not_in_tb = gl_check[gl_check["_merge"] == "left_only"]

# Accounts with a balance but no GL lines all year
lines = rf.groupby("GL_Account_Number")["Line_Count"].sum()
accounts_no_gl_lines = lines[lines == 0]

In [ ]:
tb_no_activity

In [ ]:
gl_not_in_tb

In [ ]:
accounts_no_gl_lines

In [32]:
print(f"There are {len(gl)} journal entry lines in the General Ledger.")


There are 106182 journal entry lines in the General Ledger.


In [33]:
print(f"There are {len(tb)} lines in the Trial Balance.")

There are 2756 lines in the Trial Balance.
